In [7]:
import pandas as pd

roll_no = "1024170220"
branch_name = "COPC"

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},

    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},

    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},

    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"}
]

last_digit_1 = int(roll_no[-2])
last_digit_2 = int(roll_no[-1])

categories = ["billing", "account", "general"]

cat1 = categories[last_digit_1 % 3]
cat2 = categories[last_digit_2 % 3]

personal_entry_1 = {
    "question": "what are the available support timings",
    "answer": "Our support team is available from 9 AM to 5 PM.",
    "keywords": "support hours timing",
    "category": cat1
}

personal_entry_2 = {
    "question": "how can i get a receipt for my payment",
    "answer": "A payment receipt will be sent to your registered email.",
    "keywords": "receipt payment billing",
    "category": cat2
}

faq_entries = fixed_entries + [personal_entry_1, personal_entry_2]

faq_df = pd.DataFrame(faq_entries)

print(faq_df)

                                 question  \
0                  what is the annual fee   
1                   how to reset password   
2             what are your working hours   
3                   how can i pay the fee   
4  what are the available support timings   
5  how can i get a receipt for my payment   

                                              answer                 keywords  \
0                          The annual fee is Rs 500.    fee cost price charge   
1                   Go to Settings > Reset Password.     password reset login   
2                          We are open 9 AM to 5 PM.   hours timing open time   
3         You can pay via UPI, card, or net banking.      pay payment upi fee   
4   Our support team is available from 9 AM to 5 PM.     support hours timing   
5  A payment receipt will be sent to your registe...  receipt payment billing   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  general  
5  billing  


In [8]:
def score_query(query, df):
    query_words = query.lower().split()
    results = []

    for i, row in df.iterrows():
        keywords = row["keywords"].lower().split()
        question_words = row["question"].lower().split()

        score = 0

        for word in query_words:
            if word in keywords:
                score += 1
            if word in question_words:
                score += 1

        if score > 0:
            results.append((score, i, row["question"]))

    results.sort(reverse=True)

    for score, i, question in results:
        print("Score:", score, "|", question)

    return results

query = input("Enter your query: ")
score_query(query, faq_df)

Enter your query: fee
Score: 2 | how can i pay the fee
Score: 2 | what is the annual fee


[(2, 3, 'how can i pay the fee'), (2, 0, 'what is the annual fee')]

In [9]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"]

selected_category = cat1

print("Category:", selected_category)
print(same_category(selected_category, faq_df))

Category: general
2               what are your working hours
4    what are the available support timings
Name: question, dtype: object


In [10]:
entry_number = 0

new_keyword = input("Enter a new keyword: ")

old_keywords = faq_df.loc[entry_number, "keywords"]
faq_df.loc[entry_number, "keywords"] = old_keywords + " " + new_keyword

print(faq_df)

file_name = roll_no + "_faq_data.csv"
faq_df.to_csv(file_name, index=False)

print("Saved as:", file_name)

Enter a new keyword: annual
                                 question  \
0                  what is the annual fee   
1                   how to reset password   
2             what are your working hours   
3                   how can i pay the fee   
4  what are the available support timings   
5  how can i get a receipt for my payment   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4   Our support team is available from 9 AM to 5 PM.   
5  A payment receipt will be sent to your registe...   

                       keywords category  
0  fee cost price charge annual  billing  
1          password reset login  account  
2        hours timing open time  general  
3           pay payment upi fee  billing  
4          support hours timing  general  
5    

In [11]:
category_counts = faq_df.groupby("category").size()

print(category_counts)

category
account    1
billing    3
general    2
dtype: int64


In [12]:
def score_query_with_ties(query, df):
    query_words = query.lower().split()
    results = []

    for i, row in df.iterrows():
        keywords = row["keywords"].lower().split()
        question_words = row["question"].lower().split()

        score = 0

        for word in query_words:
            if word in keywords:
                score += 1
            if word in question_words:
                score += 1

        if score > 0:
            results.append((score, i))

    if len(results) == 0:
        print("No matching entries found.")
        return

    highest_score = max(score for score, i in results)

    best_matches = []
    for score, i in results:
        if score == highest_score:
            best_matches.append(i)

    if len(best_matches) > 1:
        print("Tie found! Matching entries:")
    else:
        print("Best matching entry:")

    for i in best_matches:
        print("Score:", highest_score)
        print("Question:", faq_df.loc[i, "question"])
        print("Answer:", faq_df.loc[i, "answer"])
        print()

print("Tie demonstration:")
score_query_with_ties("fee", faq_df)

print("Non-tie demonstration:")
score_query_with_ties("password", faq_df)

Tie demonstration:
Tie found! Matching entries:
Score: 2
Question: what is the annual fee
Answer: The annual fee is Rs 500.

Score: 2
Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.

Non-tie demonstration:
Best matching entry:
Score: 2
Question: how to reset password
Answer: Go to Settings > Reset Password.

